In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/sem6/mlc/mlc_project
First sys.path entry: /Users/rithvik/Documents/sem6/mlc/mlc_project


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from src.baselines import wmmse_pf_power_allocation, wmmse_power_allocation, fractional_programming_power_allocation, max_min_power_allocation
from src.performance import calculate_sinr_and_rate
from src.dynamic_channel import DynamicChannelSimulator

In [3]:
def calculate_jains_fairness(rates):
    """
    Calculates Jain's Fairness Index for an array of user data rates.
    Formula: (Sum of rates)^2 / (N * Sum of squared rates)
    Returns a value between 1/N (worst fairness) and 1.0 (perfect equality).
    """
    # Prevent division by zero
    if np.sum(rates) == 0:
        return 0.0
    
    n = len(rates)
    numerator = np.sum(rates)**2
    denominator = n * np.sum(rates**2)
    return numerator / denominator

In [4]:
num_users_list = [4, 8, 16, 32, 64]
P_MAX = 10.0      
NOISE_VAR = 0.1   
BANDWIDTH = 1e6   
NUM_TTI = 10
EMA_BETA = 0.2

In [5]:
sum_rate_fp = []
sum_rate_wmmse = []
sum_rate_maxmin = []
sum_rate_wmmse_pf = []

jains_fp = []
jains_wmmse = []
jains_maxmin = []
jains_wmmse_pf = []

In [ ]:
for num_user in num_users_list:
    env = DynamicChannelSimulator(num_user, user_speed_kmh=60)
    historical_rates_pf = np.ones(num_user) * 1e-6

    print(f"\n{'='*30}\nEvaluating for {num_user} users\n{'='*30}")

    for tti in range(1, NUM_TTI):
        print(f"\n[{'='*15} TTI {tti + 1} {'='*15}]")
        H = env.step()

        fp_powers = fractional_programming_power_allocation(H, P_MAX, NOISE_VAR)
        fp_rates = calculate_sinr_and_rate(H, fp_powers, NOISE_VAR, BANDWIDTH)
        sum_rate_fp.append(np.mean(fp_rates))

        wmmse_powers = wmmse_power_allocation(H, P_MAX, NOISE_VAR)
        wmmse_rates = calculate_sinr_and_rate(H, wmmse_powers, NOISE_VAR, BANDWIDTH)
        sum_rate_wmmse.append(np.mean(wmmse_rates))

        max_min_powers = max_min_power_allocation(H, P_MAX, NOISE_VAR)
        max_min_rates = calculate_sinr_and_rate(H, max_min_powers, NOISE_VAR, BANDWIDTH)
        sum_rate_max_min.append(np.mean(max_min_rates))

        raw_weights = 1.0 / historical_rates_pf
        dynamic_weights = raw_weights / np.sum(raw_weights) * NUM_USERS

        wmmse_pf_powers = wmmse_pf_power_allocation(H, P_MAX, NOISE_VAR, dynamic_weights)
        wmmse_pf_rates = calculate_sinr_and_rate(H, wmmse_pf_powers, NOISE_VAR, BANDWIDTH)
        sum_rate_wmmse_pf.append(np.mean(wmmse_pf_rates))

        jains_fp.append(calculate_jains_fairness(fp_rates))
        jains_wmmse.append(calculate_jains_fairness(wmmse_rates))
        jains_maxmin.append(calculate_jains_fairness(max_min_rates))
        jains_wmmse_pf.append(calculate_jains_fairness(wmmse_pf_rates))

        historical_rates_pf = (1 - EMA_BETA) * historical_rates_pf + EMA_BETA * pf_rates

NameError: name 'NUM_USERS' is not defined